## Multi-Agent Pattern

Notes from Andrew Ng's Agentic AI course — last of the four patterns, and the one that feels most like actual software architecture.

The starting point is pretty intuitive. Some tasks just aren't one-person jobs. Andrew Ng uses this table in the course and I think it lands well:

| Task | Team needed |
|---|---|
| Create marketing assets | Researcher + Graphic Designer + Writer |
| Write a research article | Researcher + Statistician + Lead Writer + Editor |
| Prepare a legal case | Associate + Paralegal + Investigator |

You wouldn't hire one person and ask them to do all of that. Not because one person couldn't try, but because the roles require different expertise, different tools, and — practically — different ways of thinking. Multi-agent systems are just applying that same intuition to LLM agents.

What I found useful about framing it this way: it makes the design decisions feel natural. Of course the researcher needs web search. Of course the writer doesn't. Of course there needs to be something coordinating them.

### Why not just one agent with more tools?

Fair question — and for simple tasks, one agent really is fine. But there are a few places where a single agent starts to struggle as tasks get bigger.

**Context window.** If one agent is doing the research, designing the visuals, AND writing the final copy, its context fills up fast. By the time it's writing, the early research is either truncated or lost. Each agent in a multi-agent system gets a clean, focused context.

**Too many tools = worse decisions.** Give one agent 15 tools and it'll occasionally pick the wrong one, especially when tools have overlapping purposes. Keeping each agent's toolset small and relevant to its role reduces that. The researcher only has web search. The designer only has image tools. Fewer options, fewer wrong calls.

**Parallelization.** A single agent is inherently sequential — it can't do two things at once. Some steps genuinely don't depend on each other and could run simultaneously. Multi-agent systems can do that; a single agent can't.

**Specialization through the system prompt.** This one's subtle but important. An agent whose entire system prompt is "you are a fact checker, this is your only job" is noticeably better at fact checking than a general-purpose agent asked to fact-check as one of many tasks. The focused identity matters.

### The running example: marketing team

The course uses the same marketing team example throughout — I'll keep using it here so everything stays comparable across patterns.

Task: *"Create a summer marketing campaign for sunglasses."*

Three agents:

**Researcher** — analyze market trends, research competitors. Tool: web search.

**Graphic Designer** — create data visualizations and artwork. Tools: image generation, image manipulation, code execution for charts.

**Writer** — turn the research and visuals into marketing copy and a final report. Tools: none.

The writer having zero tools is deliberate. It doesn't need to look anything up — its job is purely synthesis. This is actually a design principle worth keeping: don't give agents tools they don't need. It's not just efficiency — it shapes what the agent thinks it *is*.

The four patterns below all use these same three agents. What changes is how they communicate.

### The four communication patterns

This is the main thing the course covers on multi-agent systems — how agents talk to each other. There are four patterns, ranging from "very simple" to "very flexible but hard to control."

---

**1. Linear (sequential pipeline)**

```
user → researcher → graphic designer → writer → final report
```

Each agent finishes and hands off to the next. Nothing clever happening — it's literally just a chain. The researcher's output becomes the designer's input, the designer's output feeds the writer.

Simple to implement, easy to debug. The obvious downside is that it's completely serial — no parallelism, and there's no way to go back if step 2 produces something that makes step 3 impossible.

---

**2. Orchestrator (planning with multiple agents)**

```
user → marketing manager (LLM) → generates plan:
    1. Ask researcher to research trends
    2. Ask graphic designer to create ad images
    3. Ask writer to create report
    4. Review report
→ executes plan step by step
```

A dedicated "manager" agent coordinates the others. It sees the full task, produces a plan, and routes work to the right specialist at each step. The specialists don't need to know about the big picture — they just do their bit and return a result.

The important difference from linear: the manager can adapt. If the researcher comes back with nothing useful, the manager can decide to try a different approach before calling the designer. A fixed pipeline can't do that.

---

**3. Deeper hierarchy**

```
        marketing manager
       /        |          \
 researcher  designer    writer
   /    \
web_researcher  fact_checker
```

Sub-agents can themselves have sub-agents. The marketing manager talks to the researcher, the researcher internally coordinates a web researcher and fact checker, and hands back a clean result. The marketing manager has no idea this happened.

This is useful when one of your "roles" is itself complex enough to warrant a team. The nice thing is the encapsulation — you can swap out what's happening inside the researcher team without the marketing manager needing to change at all.

---

**4. All-to-all**

```
  marketing manager ←→ researcher
         ↕                  ↕
  graphic designer  ←→   writer
```

Every agent can talk to every other agent. The designer can ask the researcher a follow-up question directly. The writer can flag something back to the manager. It's less of a pipeline and more like a shared workspace.

Most flexible, hardest to reason about. Without some kind of termination logic, agents can go in circles. This one needs careful design — the shared board implementation below is one way to handle it.

### Setup

`call_agent` is the only primitive we need — everything else is just how we wire multiple calls together.

In [ ]:
import json
from typing import Any, Dict, List, Optional
from openai import OpenAI

client = OpenAI()

def call_agent(system_prompt: str, user_message: str, model: str = "gpt-4o") -> str:
    """Single LLM call — the basic building block for all agent types below."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content

print("Setup complete.")

### Pattern 1: Linear pipeline

Researcher → designer → writer. Each step gets the previous step's full output as its input. That's it.

I like starting with this one because the code is so short. If you squint at more complex patterns and think "surely there's a simpler way" — this is the simpler way. It works well more often than you'd expect.

In [ ]:
RESEARCHER_PROMPT = """
You are a market researcher. Your job is to analyze trends and research competitors.
You have access to web search (simulated here). Given a task, produce a detailed
research summary covering: current market trends, key competitors, and key insights.
Be specific and factual. Your output will be passed to a graphic designer and writer.
""".strip()

DESIGNER_PROMPT = """
You are a graphic designer for marketing campaigns. You receive research from a researcher
and produce a visual content plan: what charts/visualizations to create, what artwork concepts
to develop, and descriptions of each asset. Be specific — describe colors, styles, content.
Your output will be passed to a writer who will incorporate it into the final report.
""".strip()

WRITER_PROMPT = """
You are a marketing copywriter. You receive research findings and a visual content plan
and produce the final marketing report. Write clearly and engagingly.
Structure it as: Executive Summary, Market Opportunity, Campaign Concept, Visual Assets, Next Steps.
""".strip()


def linear_pipeline(task: str) -> Dict[str, str]:
    """
    Sequential pipeline: researcher → designer → writer.
    Each step receives the previous step's full output.
    """
    print("[Step 1] Researcher working...")
    research = call_agent(RESEARCHER_PROMPT, task)
    print(f"Research output ({len(research)} chars)\n")

    print("[Step 2] Graphic designer working...")
    design_brief = call_agent(
        DESIGNER_PROMPT,
        f"Research findings:\n{research}\n\nOriginal task: {task}"
    )
    print(f"Design brief output ({len(design_brief)} chars)\n")

    print("[Step 3] Writer working...")
    final_report = call_agent(
        WRITER_PROMPT,
        f"Research:\n{research}\n\nVisual content plan:\n{design_brief}\n\nOriginal task: {task}"
    )
    print(f"Final report ({len(final_report)} chars)\n")

    return {"research": research, "design_brief": design_brief, "final_report": final_report}


task = "Create a summer marketing campaign for sunglasses"
results = linear_pipeline(task)

print("=" * 60)
print("FINAL REPORT:")
print("=" * 60)
print(results["final_report"])

### Pattern 2: Orchestrator

Now there's a dedicated manager agent that generates a plan first, then coordinates execution. The subagents (researcher, designer, writer) don't change at all — same prompts as above. What changes is that a manager is deciding when to call them and what to tell them.

The system prompt for the orchestrator is where most of the design work lives. It needs to know who's on the team, what each person does, and be told to return a structured plan. The `depends_on` field per step is how we handle ordering — step 3 won't run until its dependencies are complete.

In [ ]:
ORCHESTRATOR_PROMPT = """
You are a marketing manager coordinating a team of specialist agents.

Your team:
- researcher: Analyzes market trends and researches competitors. Has web search access.
- graphic_designer: Creates data visualizations, charts, and artwork concepts. Has image tools.
- writer: Transforms research and visual plans into marketing copy and reports. No tools needed.

Given a task, produce a step-by-step plan to complete it using your team.
Return JSON with a 'plan' array. Each step: step, agent, instruction, depends_on (list of step numbers).
Keep instructions specific — the agent should know exactly what you need from them.
""".strip()


def orchestrate(task: str) -> str:
    """
    Orchestrator pattern:
    1. Marketing manager generates a plan
    2. We execute each step in order, passing relevant outputs forward
    3. Marketing manager synthesizes a final answer from all results
    """
    # Phase 1: planning
    print("[Orchestrator] Generating plan...")
    plan_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": ORCHESTRATOR_PROMPT},
            {"role": "user", "content": task},
        ],
        response_format={"type": "json_object"},
    )
    plan = json.loads(plan_response.choices[0].message.content).get("plan", [])

    print("Plan:")
    for step in plan:
        deps = step.get("depends_on", [])
        print(f"  Step {step['step']} [{step['agent']}]: {step['instruction'][:80]}...")
        if deps:
            print(f"    depends on: steps {deps}")
    print()

    # Phase 2: execution
    agent_prompts = {
        "researcher": RESEARCHER_PROMPT,
        "graphic_designer": DESIGNER_PROMPT,
        "writer": WRITER_PROMPT,
    }

    step_outputs: Dict[int, str] = {}

    for step in plan:
        step_num = step["step"]
        agent = step["agent"]
        instruction = step["instruction"]
        depends_on = step.get("depends_on", [])

        print(f"[Step {step_num}] Calling {agent}...")

        # Build the message including outputs from all dependency steps
        message = f"Your task: {instruction}\n\nOriginal request: {task}"
        if depends_on:
            message += "\n\nInputs from previous steps:"
            for dep in depends_on:
                if dep in step_outputs:
                    message += f"\n\nStep {dep} output:\n{step_outputs[dep]}"

        prompt = agent_prompts.get(agent, "You are a helpful assistant.")
        output = call_agent(prompt, message)
        step_outputs[step_num] = output
        print(f"  Done ({len(output)} chars)\n")

    # Phase 3: final synthesis by the orchestrator
    print("[Orchestrator] Synthesizing final output...")
    all_outputs = "\n\n".join(
        f"Step {k} ({plan[k-1]['agent']}):\n{v}" for k, v in step_outputs.items()
    )
    final = call_agent(
        "You are a marketing manager. Review all the work your team has produced and write a concise final summary of the campaign plan.",
        f"Original task: {task}\n\nTeam outputs:\n{all_outputs}"
    )
    return final


result = orchestrate("Create a summer marketing campaign for sunglasses")
print("=" * 60)
print("ORCHESTRATED RESULT:")
print("=" * 60)
print(result)

### Pattern 3: Deeper hierarchy

The researcher now has its own internal team — a web researcher and a fact checker. From the marketing manager's perspective nothing changes: it still calls "researcher" and gets back a research summary. It has no idea a sub-team was involved.

```
        marketing manager
       /        |          \
 researcher  designer    writer
   /    \
web_researcher  fact_checker
```

The web researcher gathers raw data, the fact checker flags anything questionable, and the lead researcher synthesizes a clean output from both. The marketing manager sees none of that — just the final research result.

This encapsulation is the main reason to use this pattern. If you later decide the researcher needs a citation checker or a statistician sub-agent, you add it inside `researcher_team()` and nothing else needs to change.

In [ ]:
# Sub-agents under the researcher

WEB_RESEARCHER_PROMPT = """
You are a web researcher. Your job is to find raw information, data, and sources on a given topic.
Produce a detailed list of findings: statistics, trends, quotes, and source descriptions.
Don't synthesize yet — just gather. The fact checker and lead researcher will refine this.
""".strip()

FACT_CHECKER_PROMPT = """
You are a fact checker. You receive raw research findings and your job is to:
- Flag any claims that seem questionable or need verification
- Remove or caveat anything that looks like speculation presented as fact
- Mark the most solid, well-supported findings as verified
Return a cleaned version of the findings with confidence levels noted.
""".strip()

LEAD_RESEARCHER_PROMPT = """
You are a lead researcher managing a web researcher and fact checker.
You receive their combined outputs and synthesize them into a clean, well-structured
research summary. Remove redundancy, organize by theme, and make it ready for a
marketing team to act on.
""".strip()


def researcher_team(task: str) -> str:
    """
    The researcher agent internally orchestrates its own sub-team.
    The marketing manager just calls this function and gets back clean research.
    It doesn't know or care that two sub-agents were used internally.
    """
    print("  [researcher > web_researcher] Gathering raw data...")
    raw_findings = call_agent(WEB_RESEARCHER_PROMPT, task)

    print("  [researcher > fact_checker] Verifying findings...")
    verified_findings = call_agent(
        FACT_CHECKER_PROMPT,
        f"Raw findings to check:\n{raw_findings}"
    )

    print("  [researcher > lead_researcher] Synthesizing...")
    final_research = call_agent(
        LEAD_RESEARCHER_PROMPT,
        f"Raw findings:\n{raw_findings}\n\nFact-checked version:\n{verified_findings}\n\nOriginal task: {task}"
    )

    return final_research


def hierarchical_pipeline(task: str) -> str:
    """
    Same linear flow as Pattern 1, but the researcher now has an internal team.
    Marketing manager still just sees: researcher → designer → writer.
    """
    print("[Step 1] Researcher team working...")
    research = researcher_team(task)
    print(f"  Research ready ({len(research)} chars)\n")

    print("[Step 2] Graphic designer working...")
    design_brief = call_agent(
        DESIGNER_PROMPT,
        f"Research findings:\n{research}\n\nTask: {task}"
    )
    print(f"  Design brief ready ({len(design_brief)} chars)\n")

    print("[Step 3] Writer working...")
    final_report = call_agent(
        WRITER_PROMPT,
        f"Research:\n{research}\n\nVisual plan:\n{design_brief}\n\nTask: {task}"
    )
    print(f"  Final report ready ({len(final_report)} chars)\n")

    return final_report


report = hierarchical_pipeline("Create a summer marketing campaign for sunglasses")
print("=" * 60)
print("HIERARCHICAL RESULT:")
print("=" * 60)
print(report)

### Pattern 4: All-to-all

Every agent can talk to every other agent. No fixed chain. The designer can ask the researcher a clarifying question mid-way. The writer can flag something back to the manager before finishing.

I'll be upfront — this is the hardest one to implement cleanly. The shared board approach below (every agent reads and posts to a common message history) is one way to do it. It works but you have to be intentional about when to stop, otherwise agents just keep reacting to each other indefinitely.

In the course, the all-to-all diagram shows bidirectional arrows between all four nodes (manager, researcher, designer, writer). The practical implication is that you lose the clean sequencing of the other patterns and gain genuine flexibility in how the work unfolds.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class SharedBoard:
    """
    Shared message board all agents can read from and write to.
    Each message has a sender and content — agents read the full history
    to understand what's been done and what's needed next.
    """
    messages: List[Dict[str, str]] = field(default_factory=list)

    def post(self, sender: str, content: str) -> None:
        self.messages.append({"sender": sender, "content": content})
        print(f"  [{sender}] posted ({len(content)} chars)")

    def read_all(self) -> str:
        """Returns full board history as a single string for context."""
        return "\n\n".join(
            f"[{msg['sender']}]: {msg['content']}" for msg in self.messages
        )


def agent_turn(name: str, system_prompt: str, board: SharedBoard, instruction: str) -> str:
    """
    Run one turn for an agent. It reads the full board history as context,
    gets a specific instruction for this turn, and posts its output back.
    """
    board_context = board.read_all()
    message = f"{instruction}\n\nTeam board so far:\n{board_context}" if board_context else instruction
    output = call_agent(system_prompt, message)
    board.post(name, output)
    return output


def all_to_all_workflow(task: str) -> str:
    """
    All-to-all pattern using a shared board.

    Agents work in turns but each one sees everything posted so far.
    The marketing manager posts the task, agents contribute, manager synthesizes.
    In a fuller implementation, agents could request each other by name mid-flow.
    """
    board = SharedBoard()

    # Manager posts the brief
    board.post("marketing_manager", f"Team task: {task}\n\nPlease contribute your expertise.")

    print("\n[Round 1] Researcher — initial research")
    agent_turn(
        "researcher", RESEARCHER_PROMPT, board,
        "Read the team board and provide your market research findings."
    )

    print("\n[Round 2] Graphic designer — visual plan based on research")
    agent_turn(
        "graphic_designer", DESIGNER_PROMPT, board,
        "Read the team board including the researcher's findings and propose a visual content plan."
    )

    print("\n[Round 3] Researcher — clarification requested by designer (simulated)")
    agent_turn(
        "researcher", RESEARCHER_PROMPT, board,
        "The graphic designer may need additional data. Review the board and add any clarifications or extra data points that would help the visual plan."
    )

    print("\n[Round 4] Writer — final report using everything on the board")
    agent_turn(
        "writer", WRITER_PROMPT, board,
        "Read everything on the team board and write the final marketing campaign report."
    )

    print("\n[Manager] Reviewing and signing off...")
    final = call_agent(
        "You are a marketing manager. Review your team's work and write a short executive sign-off summarizing the campaign.",
        f"Original task: {task}\n\nTeam board:\n{board.read_all()}"
    )

    return final


result = all_to_all_workflow("Create a summer marketing campaign for sunglasses")
print("\n" + "=" * 60)
print("MANAGER SIGN-OFF:")
print("=" * 60)
print(result)

### System prompts are where the real work is

Something the course touches on that I think is easy to underestimate: the quality of your system prompts matters more than the orchestration pattern you pick.

A general agent with a vague system prompt ("you are a helpful assistant") asked to check facts will produce vague, conversational output. An agent whose system prompt is entirely focused on fact-checking — specific format, specific criteria, specific output structure — will do it much better.

This sounds obvious but the implication is that most of the engineering effort in a multi-agent system isn't the routing code. It's writing and iterating on the system prompts for each role. The code below shows this directly — same input, same model, completely different output quality just from the system prompt.

In [ ]:
# Illustrating the difference between a general vs specialized agent on the same task

GENERAL_PROMPT = "You are a helpful assistant."

SPECIALIZED_FACT_CHECKER_PROMPT = """
You are a fact checker. You do one thing: evaluate claims for accuracy.
For each claim in the text, respond with:
- VERIFIED: if it's well-supported
- UNCERTAIN: if it needs a source or is hard to verify
- FLAG: if it looks wrong or misleading

Be brief. Don't rewrite the text. Just annotate each claim.
""".strip()

sample_text = """
The global sunglasses market is worth $20 billion and growing at 8% annually.
Polarized lenses account for 60% of all sunglasses sales.
Summer is the highest sales period, with July being the single best month.
Millennials prefer sport-style frames over classic aviators.
UV protection is now required by law in the EU for all sunglasses sold.
""".strip()

print("=" * 50)
print("General agent response:")
print("=" * 50)
general_response = call_agent(GENERAL_PROMPT, f"Check this text for accuracy:\n{sample_text}")
print(general_response)

print("\n" + "=" * 50)
print("Specialized fact checker response:")
print("=" * 50)
specialized_response = call_agent(SPECIALIZED_FACT_CHECKER_PROMPT, sample_text)
print(specialized_response)

### Parallelization

Not covered as its own slide in the course but it's kind of implied by the whole multi-agent framing. Once you have separate agents, steps that don't depend on each other can run at the same time.

In the marketing example, trend analysis and competitor analysis are independent — neither needs the other's output to start. Running them in sequence is just wasted time. With `asyncio.gather` they run simultaneously and you get both results back before calling the designer.

This is actually one of the bigger practical wins from multi-agent systems in production — tasks that took 3 minutes sequentially can drop to 60-90 seconds just by parallelizing the independent steps.

In [ ]:
import asyncio
import time


async def call_agent_async(system_prompt: str, user_message: str) -> str:
    """Async wrapper around the OpenAI call for parallel execution."""
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(
        None,
        lambda: call_agent(system_prompt, user_message)
    )


TREND_ANALYST_PROMPT = """
You are a trend analyst. Research current consumer trends in eyewear and outdoor fashion.
Focus on what's popular right now, what's declining, and what's emerging.
""".strip()

COMPETITOR_ANALYST_PROMPT = """
You are a competitive intelligence analyst. Research the top sunglasses brands —
their positioning, pricing, target demographics, and recent marketing campaigns.
""".strip()


async def parallel_research(task: str) -> Dict[str, str]:
    """
    Run two independent research agents at the same time.
    Trend analysis and competitor analysis don't depend on each other,
    so there's no reason to do them sequentially.
    """
    print("Running trend analyst and competitor analyst in parallel...")
    start = time.time()

    trend_task = call_agent_async(TREND_ANALYST_PROMPT, task)
    competitor_task = call_agent_async(COMPETITOR_ANALYST_PROMPT, task)

    trends, competitors = await asyncio.gather(trend_task, competitor_task)

    elapsed = time.time() - start
    print(f"Both complete in {elapsed:.1f}s (vs ~{elapsed * 1.8:.1f}s sequentially)")

    return {"trends": trends, "competitors": competitors}


async def pipeline_with_parallel_research(task: str) -> str:
    """Full pipeline where the research phase runs two agents in parallel."""
    # Parallel research phase
    research_results = await parallel_research(task)

    # Merge research outputs for downstream agents
    combined_research = (
        f"TREND ANALYSIS:\n{research_results['trends']}\n\n"
        f"COMPETITOR ANALYSIS:\n{research_results['competitors']}"
    )

    # Sequential from here — designer needs research, writer needs both
    print("\n[Sequential] Graphic designer...")
    design_brief = call_agent(
        DESIGNER_PROMPT,
        f"Research:\n{combined_research}\n\nTask: {task}"
    )

    print("[Sequential] Writer...")
    final_report = call_agent(
        WRITER_PROMPT,
        f"Research:\n{combined_research}\n\nVisual plan:\n{design_brief}\n\nTask: {task}"
    )

    return final_report


report = await pipeline_with_parallel_research("Create a summer marketing campaign for sunglasses")
print("\n" + "=" * 60)
print("PARALLEL RESEARCH PIPELINE RESULT:")
print("=" * 60)
print(report)

### Which pattern to use

The course doesn't spell this out explicitly but based on the examples here's how I'd think about it:

**Linear** — steps are naturally sequential, each one fully depends on the previous, and you don't expect to need to adapt mid-way. Start here. It's easy to debug and easy to extend later.

**Orchestrator** — the task has messier dependencies, or you want the flexibility to adapt based on intermediate results. Also good when you want to inspect the plan before execution — useful for anything with irreversible steps.

**Deeper hierarchy** — one of your top-level roles is itself complex enough to need internal coordination. Use this to encapsulate that complexity rather than exposing it to the top-level manager. Good for systems that will grow over time.

**All-to-all** — you genuinely can't predict the interaction pattern upfront, or the task needs agents to react to each other dynamically. Be honest about the cost: this one is significantly harder to debug and test. Make sure the flexibility is worth it before going here.

Most real production systems end up as some combination — linear at the top, with an orchestrated or hierarchical sub-workflow inside one of the steps.

### Things the course doesn't cover but come up quickly in practice

**Shared memory across runs.** Every agent in these examples only knows what it's explicitly given in the current run. If you want a researcher that actually gets better over time — builds up a knowledge base, remembers what it searched last week — you need a shared store (vector DB, key-value cache) that agents can read from and write to independently. None of the course examples touch this.

**Error propagation is silent and confident.** If the researcher returns garbage, the designer will confidently produce a visual plan based on garbage, and the writer will confidently write a report based on that. Nobody flags it. Each agent needs to either validate its inputs or the orchestrator needs to check outputs before passing them forward. Easy to miss until you see it happen.

**Agent identity in all-to-all.** In the shared board pattern, how does the writer actually know the message tagged "researcher" came from the researcher agent and not from a hallucination? For small demo systems it doesn't matter much. For anything larger with real consequences, structured message formats with verified sender IDs matter.

**Costs multiply fast.** Five agents where each calls the others can burn through tokens surprisingly quickly. Worth tracking per-agent costs early and figuring out which outputs could be cached or reused across similar tasks.

**Termination in all-to-all needs explicit design.** Without it, agents will keep responding to each other. Max rounds is the simplest fix. A cleaner approach is having the orchestrator check after each round whether the goal has been met and explicitly stop if it has.

### Quick reference

```
4 communication patterns:

  1. LINEAR          A → B → C
                     Each agent hands off to the next.
                     Use when: steps are sequential, task is well-defined.

  2. ORCHESTRATOR    manager plans → routes to specialists → synthesizes
                     Manager can adapt based on intermediate results.
                     Use when: dependencies are messy, or you need a reviewable plan.

  3. HIERARCHY       manager → mid-level agents → specialist workers
                     Complexity is encapsulated at each level.
                     Use when: a role is itself complex enough to need a sub-team.

  4. ALL-TO-ALL      every agent can talk to every other agent
                     Most flexible, needs explicit termination logic.
                     Use when: interaction pattern can't be predicted upfront.

Why multiple agents at all:
  each agent gets a clean focused context   (not one bloated shared one)
  small tool sets → fewer wrong tool calls
  independent steps can run in parallel
  encapsulation: internals can change without affecting interfaces

What bites you in practice:
  garbage in → garbage out, silently and confidently
  all-to-all without a stop condition runs forever
  token costs multiply across many agents
  no persistent memory = agents don't improve across runs
```